In [40]:
import os
print(os.getcwd())

/Users/username/LISA/DVR Conversation/Agentic_DVR/backend


In [ ]:
from dotenv import load_dotenv
from datetime import datetime, timezone, timedelta

load_dotenv()

from utils.auth_util import AuthManager
auth_manager = AuthManager()

# url = "/v1/fleets/lmfleet003/drivers/mithun/trips/t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z"
url = "/v1/fleets/lmfleet003/drivers/mithun/trips/t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z"

trips = auth_manager.make_api_request(
    client_id="lmdemotsp",
    endpoint=url,
    params = {
        "includeViolations" : True,
        "includePathInfo" : True,
        "includeUploadRequests" : True
    }
)

In [42]:
trips

{'asset': {'assetId': 'mithun-k245-fin',
  'dutyType': 'light',
  'packages': ['DVR',
   'EDVR',
   'G-sensor',
   'ADAS',
   'DMS',
   'LIVE_STREAMING',
   'LIVE_TELEMATICS'],
  'assetName': 'Mithun Mitac Fin',
  'cameraType': 'RIDECAM_PLUS',
  'plusPackages': ['DVR',
   'EDVR',
   'G-sensor',
   'ADAS',
   'DMS',
   'LIVE_STREAMING',
   'LIVE_TELEMATICS'],
  'ridecamPlusPlan': 'ADVANCED'},
 'device': {'model': 'MiTAC CDR', 'deviceId': '869409069759846'},
 'tripId': 't_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z',
 'dvrMode': 'FULL',
 'endTime': '2026-09-25T10:44:23.807Z',
 'fleetId': 'lmfleet003',
 'ongoing': False,
 'division': '_UNIDENTIFIED',
 'driverId': 'mithun',
 'duration': 3499,
 'startTime': '2026-09-25T09:46:04.333Z',
 'driverName': 'Mithun',
 'dvrEnabled': True,
 'endTimeUTC': '2026-09-25T05:14:23.807Z',
 'eventCount': {'total': 15,
  'Traffic-Speed-Violated': 3,
  'Cornering': 0,
  'Traffic-STOP-Sign-Violated': 0,
  'Harsh-Braking': 0,
  'Harsh-Acceleration': 0,
  'Tail-Gating-Detec

In [43]:
trip_info={}
def simplify_response(trip):
    """
    Simplifies the trip response from the API to a more manageable format.
    Includes trip details, asset information, driver information, and violations.
    """
    trip_info['tripId'] = trip.get('tripId')
    trip_info['startTime'] = trip.get('startTime')
    trip_info['endTime'] = trip.get('endTime')
    trip_info['duration'] = trip.get('duration')
    trip_info['distance'] = trip.get('distance')
    trip_info['averageSpeed'] = trip.get('averageSpeed')
    trip_info['maxSpeed'] = trip.get('maxSpeed')
    trip_info['asset_info'] = {
        'assetId': trip.get('asset', {}).get('assetId'),
        'assetName': trip.get('asset', {}).get('assetName'),
        'cameraType': trip.get('asset', {}).get('cameraType'),
        'dutyType': trip.get('asset', {}).get('dutyType')}
    trip_info['fleetId'] = trip.get('fleetId')
    trip_info['driverId'] = trip.get('driverId')
    trip_info['driverName'] = trip.get('driverName')
    trip_info['eventCount'] = trip.get('eventCount')
    trip_info['violations'] = [{
        'event_index' : violation.get('eventIndex'),
        'event_type' : violation.get('eventType'),
        ' ' : violation.get('timestampUTC'),
        'event_location' : {
            'latitude' : violation.get('latitude', {}), 
            'longitude' : violation.get('longitude', {})},
        } for violation in trip.get('violations', [])]
    return trip_info


def get_trip_route(trip):
    route = []
    for point in trip.get('pathInfo', []):
        route.append({
            'timeOfDay': point.get('timeOfDay'),
            'timeUTC' : point.get('timestampUTC'),
            'latitude': point.get('latitude'),
            'longitude': point.get('longitude')
        })
    summarised_route = []
    for i in range(0, len(route), 100):
        summarised_route.append(route[i])
    return {
        'route': route,
        'summarised_route': summarised_route
    }



def get_trip_upload_requests(trip):
    upload_requests = []
    for request in trip.get('uploadRequests', []):
        upload_requests.append({
            'requestId': request.get('requestId'),
            'status': request.get('status'),
            'timestamp': request.get('createdAt')})
    return upload_requests
    


In [44]:
print(get_trip_route(trips).get('summarised_route'))

[{'timeOfDay': '04:16:04', 'timeUTC': 1790309764905, 'latitude': 13.03028, 'longitude': 77.53765}, {'timeOfDay': '04:17:47', 'timeUTC': 1790309867898, 'latitude': 13.03097, 'longitude': 77.54045}, {'timeOfDay': '04:19:27', 'timeUTC': 1790309967895, 'latitude': 13.03651, 'longitude': 77.54154}, {'timeOfDay': '04:21:07', 'timeUTC': 1790310067930, 'latitude': 13.04133, 'longitude': 77.54619}, {'timeOfDay': '04:22:47', 'timeUTC': 1790310167900, 'latitude': 13.04361, 'longitude': 77.55282}, {'timeOfDay': '04:24:27', 'timeUTC': 1790310267897, 'latitude': 13.04437, 'longitude': 77.56221}, {'timeOfDay': '04:26:07', 'timeUTC': 1790310367901, 'latitude': 13.04724, 'longitude': 77.57179}, {'timeOfDay': '04:27:48', 'timeUTC': 1790310468904, 'latitude': 13.0447, 'longitude': 77.58306}, {'timeOfDay': '04:29:28', 'timeUTC': 1790310568904, 'latitude': 13.04316, 'longitude': 77.58619}, {'timeOfDay': '04:31:08', 'timeUTC': 1790310668902, 'latitude': 13.0432, 'longitude': 77.59008}, {'timeOfDay': '04:32:

In [45]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [46]:
from pydantic import BaseModel

class Cross_Question_Response(BaseModel):
    cross_question: str = None
    options: list[str] = []

## Two tools one to get info from the violations and the other to get info from the route   

prompt = """
allow the user to select the exact place
cross question about the duration of the DVR, if more than 3 minutes ask the user if they want to see the entire timelapse or just a snippet as DVR max allowance is 3 min 
cross question about the duration of the timelapse
get timeslot using the information of the violations(time), or the location
"""
class Time_Slot_Response(BaseModel):
    cross_question: Cross_Question_Response = None
    start_time: str = None
    end_time: str = None

# plan for creating the agent that handels the find time slot --
# two tools one to get info from the violations and the other to get info from the route
#
# for the route : 
# get the location(extract from the user query) --> 
# geocode place (get a list of lat long) --> 
# check if the place is within the route or not (if not then ask the user to select a place from the route) --> 
# get the time slot using the information of the violations(time), or the location

In [47]:
# to get the coordinate of the place
import os
import requests
from dotenv import load_dotenv
load_dotenv()

GEOAPIFY_URL = "https://api.geoapify.com/v1/geocode/search"
AREA_TYPES = {"city", "suburb", "district", "county", "locality"}

def geocode_place(place_name):
    params = {
        "text": place_name,
        "format": "json",
        "limit": 5,
        "bias": "countrycode:none",                 # prefer Indian results
        "apiKey": os.environ["GEOAPIFY_API_KEY"],
    }
    resp = requests.get(GEOAPIFY_URL, params=params, timeout=5)
    resp.raise_for_status()

    return [
        {
            "name": r.get("formatted", ""),
            "lat": r["lat"],
            "lon": r["lon"],
            "types": ["locality"] if r.get("result_type") in AREA_TYPES
                     else [r.get("result_type", "")],   # keeps threshold_for working
        }
        for r in resp.json().get("results", [])
    ]


In [48]:
candidates = geocode_place("Bhuvanahalli")
candidates

[{'name': 'Bhuvanahalli, Turuvekere taluk, India',
  'lat': 13.1890012,
  'lon': 76.6321546,
  'types': ['locality']},
 {'name': 'Bhuvanahalli - 560300, KA, India',
  'lat': 13.2176918,
  'lon': 77.6816863,
  'types': ['locality']},
 {'name': 'Bhuvanahalli - 571102, KA, India',
  'lat': 12.4429398,
  'lon': 76.0857486,
  'types': ['locality']},
 {'name': 'Bhuvanahalli - 572139, KA, India',
  'lat': 13.7582287,
  'lon': 76.8544684,
  'types': ['locality']},
 {'name': 'Bhuvanahalli - 571802, KA, India',
  'lat': 12.8716696,
  'lon': 76.6069454,
  'types': ['locality']}]

In [49]:
# function to create a bounding box around the route

import math

M_PER_DEG_LAT = 111_320          # metres in 1° of latitude

def route_bbox(route, margin_m=750):
    """
    route: [{'timeOfDay': 'HH:MM:SS', 'latitude': float, 'longitude': float}, ...]
    Returns (min_lat, max_lat, min_lon, max_lon) padded by margin_m metres, or None if route is empty.
    """
    if not route:
        return None

    lats = [p["latitude"] for p in route]
    lons = [p["longitude"] for p in route]
    min_lat, max_lat = min(lats), max(lats)
    min_lon, max_lon = min(lons), max(lons)

    lat_pad = margin_m / M_PER_DEG_LAT
    mid_lat = (min_lat + max_lat) / 2
    lon_pad = margin_m / (M_PER_DEG_LAT * math.cos(math.radians(mid_lat)))

    return (min_lat - lat_pad, max_lat + lat_pad,
            min_lon - lon_pad, max_lon + lon_pad)


def inside_bbox(lat, lon, bbox):
    min_lat, max_lat, min_lon, max_lon = bbox
    return min_lat <= lat <= max_lat and min_lon <= lon <= max_lon

In [50]:
# filter the candidates to only those that are within the bounding box of the route

def check_candidates_in_bbox(candidates, route):
    bbox = route_bbox(route)
    in_bbox = []
    for place in candidates:
        if inside_bbox(place["lat"], place["lon"], bbox):
            in_bbox.append(place)
            print(f"{place['name']} is within the route bounding box.")
        else:
            print(f"{place['name']} is NOT within the route bounding box.")
    return in_bbox


print(check_candidates_in_bbox(candidates, get_trip_route(trips).get('route')))

Bhuvanahalli, Turuvekere taluk, India is NOT within the route bounding box.
Bhuvanahalli - 560300, KA, India is within the route bounding box.
Bhuvanahalli - 571102, KA, India is NOT within the route bounding box.
Bhuvanahalli - 572139, KA, India is NOT within the route bounding box.
Bhuvanahalli - 571802, KA, India is NOT within the route bounding box.
[{'name': 'Bhuvanahalli - 560300, KA, India', 'lat': 13.2176918, 'lon': 77.6816863, 'types': ['locality']}]


In [51]:
# finding the nearest point on the route to a given lat/lon, within a threshold distance
import math

EARTH_RADIUS_M = 6_371_000

def haversine_m(lat1, lon1, lat2, lon2):
    """Distance in metres between two lat/lon points along the Earth's surface."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_RADIUS_M * math.asin(math.sqrt(a))


def nearest_route_point(route, lat, lon, max_dist_m=1150):
    """
    route: [{'timeOfDay': 'HH:MM:SS', 'latitude': float, 'longitude': float}, ...]
    Returns {"time", "dist_m", "lat", "lon"} of the closest route point,
    or None if even the closest is farther than max_dist_m.
    """
    best = None
    for p in route:
        d = haversine_m(lat, lon, p["latitude"], p["longitude"])
        if best is None or d < best["dist_m"]:
            best = {"time": p["timeOfDay"], "timeUTC": p["timeUTC"], "dist_m": d,
                    "lat": p["latitude"], "lon": p["longitude"]}

    if best is None or best["dist_m"] > max_dist_m:
        return None
    return best

def closest_point_candidate(route, candidates, threshold_m=1150):
    """
    Returns the closest candidate to the route within threshold_m,
    or None if none are close enough.
    """
    closest = None
    for c in candidates:
        point = nearest_route_point(route, c["lat"], c["lon"], max_dist_m=threshold_m)
        if point and (closest is None or point["dist_m"] < closest["point"]["dist_m"]):
            closest = {"place": c, "point": point}
    return closest

print(candidates)
final_point = closest_point_candidate(get_trip_route(trips).get('route'), candidates, threshold_m=1150)

[{'name': 'Bhuvanahalli, Turuvekere taluk, India', 'lat': 13.1890012, 'lon': 76.6321546, 'types': ['locality']}, {'name': 'Bhuvanahalli - 560300, KA, India', 'lat': 13.2176918, 'lon': 77.6816863, 'types': ['locality']}, {'name': 'Bhuvanahalli - 571102, KA, India', 'lat': 12.4429398, 'lon': 76.0857486, 'types': ['locality']}, {'name': 'Bhuvanahalli - 572139, KA, India', 'lat': 13.7582287, 'lon': 76.8544684, 'types': ['locality']}, {'name': 'Bhuvanahalli - 571802, KA, India', 'lat': 12.8716696, 'lon': 76.6069454, 'types': ['locality']}]


In [52]:
from dateutil import parser, tz

# converts the time string to utc
def to_utc_string(time_str: str):
    dt = parser.parse(time_str)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=tz.UTC)

    dt = dt.astimezone(tz.UTC)

    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

target = datetime.fromisoformat(to_utc_string("12th september 03:34:56").replace("Z", "+00:00"))

datetime.fromisoformat(to_utc_string("12th September 2026"))

datetime.datetime(2026, 9, 12, 0, 0, tzinfo=datetime.timezone.utc)

In [53]:
# finding the violation event that occurred closest to a geocoded place

def closest_point_event(trip_info, violation_name, candidates, threshold_m=500):
    """
    violations: trip_info["violations"] -> [{'event_type', 'event_timeUTC',
                'event_location': {'latitude', 'longitude'}}, ...]
    violation_name: matched event type (e.g. 'Traffic-Speed-Violated') or None for any event.
    candidates: geocoded places [{'name', 'lat', 'lon', ...}, ...]

    Returns (closest, message):
      - closest = {"place": candidate, "event": violation, "dist_m": float} for the event
        nearest to any candidate within threshold_m, message = None
      - (None, message) if no matching event occurred within threshold_m of any candidate.
    """
    violations = trip_info.get('violations')
    events = (violations if violation_name is None
              else [v for v in violations if v.get("event_type") == violation_name])
    label = violation_name or "violation"

    closest = None
    for c in candidates:
        for v in events:
            loc = v.get("event_location", {})
            lat, lon = loc.get("latitude"), loc.get("longitude")
            if not isinstance(lat, (int, float)) or not isinstance(lon, (int, float)):
                continue
            d = haversine_m(c["lat"], c["lon"], lat, lon)
            if d <= threshold_m and (closest is None or d < closest["dist_m"]):
                closest = {"place": c, "event": v, "dist_m": d}

    if closest is None:
        return None, f"No {label} event occurred within {threshold_m} m of this location."
    return closest, None

trip_info = simplify_response(trips)
def get_time_slot_from_place_violation(closest, threshold_min):
    event = closest.get('event')
    time = event.get('event_timeUTC')
    dt = datetime.fromisoformat(time.replace("Z", "+00:00"))
    delta = timedelta(minutes=threshold_min)
    
    return {
        'startTimeUTC': (dt - delta).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        'endTimeUTC': (dt + delta).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    }


print(closest_point_event(trip_info, "Traffic-Speed-Violated", candidates, threshold_m=1500))
print(closest_point_event(trip_info, "Harsh-Braking", candidates))

({'place': {'name': 'Bhuvanahalli - 560300, KA, India', 'lat': 13.2176918, 'lon': 77.6816863, 'types': ['locality']}, 'event': {'event_index': 155, 'event_type': 'Traffic-Speed-Violated', ' ': '2026-09-25T04:55:21.034Z', 'event_location': {'latitude': 13.22367, 'longitude': 77.68449}}, 'dist_m': 730.7499741747092}, None)
(None, 'No Harsh-Braking event occurred within 500 m of this location.')


In [54]:
# get the exact timestamp of that point on the route, and return it as a time slot
from datetime import datetime, timezone, timedelta

time = final_point["point"]["timeUTC"]
dt = datetime.fromtimestamp(time / 1000, tz=timezone.utc)

print(dt)

def get_time_slot_from_point(point, threshold_min=1.5):
    """
    Given a point on the route, returns a time slot around that point.
    threshold_min: half-width of the time slot in minutes.
    Returns (start_time, end_time) as datetime objects in UTC.
    """
    time = point["timeUTC"]
    dt = datetime.fromtimestamp(time / 1000, tz=timezone.utc)
    delta = timedelta(minutes=threshold_min)
    return {
        'startTimeUTC': (dt - delta).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        'endTimeUTC': (dt + delta).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    }
print(get_time_slot_from_point(final_point["point"], threshold_min=1.5))

2026-09-25 04:54:49.896000+00:00
{'startTimeUTC': '2026-09-25 04:53:19.896', 'endTimeUTC': '2026-09-25 04:56:19.896'}


In [55]:
from langchain_core.tools import tool


# when only place is mentioned, place with event is mentioned

def get_time_slot_from_place(place_name, route, violation_mentioned, violation_name, threshold_min=1.5):
    """
    Find the footage time slot for when the selected trip was at a place the user mentions.

    Purpose:
        Geocodes the place the user mentioned, checks it lies on the trip's
        route, and returns the time slot (start and end time) for the DVR
        request. If the user also mentions that a violation/event happened at
        that place, the tool looks for that violation near the place and centres
        the slot on it. Otherwise it centres the slot on the moment the vehicle
        passed closest to the place.

    When to use:
        Use when the user identifies the moment by a LOCATION, with or without
        a violation:
        - Place only:
            "get me the footage from when the truck was near Hosur"
            "DVR clip of the trip near Silk Board"
        - Place with an unnamed event:
            "an event occurred near Bhuvanahalli"
        - Place with a named violation:
            "the overspeeding near Bhuvanahalli"

    When NOT to use:
        - The user mentions a violation/event with no location, with or without
          a time ("harsh braking at 4:20 pm", "a violation occurred"). Use
          get_time_slot_from_violation instead.
        - No trip is selected yet. Fetch and select a trip first.

    Args:
        place_name (str): The place as the user said it, e.g. "Bhuvanahalli".
            Misspellings are tolerated, since geocoding resolves to the closest
            real place.
        route (list): Route points of the selected trip, each [lat, lon, time].
        violation_mentioned (bool): True if the user said a violation/event
            happened at the place, whether or not they named it. False if they
            only mentioned the place.
        violation_name (str | None): Violation as the user phrased it, e.g.
            "traffic speed violation". It is fuzzy-matched to the trip's
            violation types. Pass None if no violation was named.
        threshold_min (float): Minutes of footage on EACH side of the matched
            moment. Default 1.5, which gives a 3-minute clip.

    Returns:
        dict with a "status" key:
        - "success": the place was matched on the route. Includes "place"
          (resolved place name) and "timeSlot" (start and end time of the clip).
        - "not_found": the place could not be geocoded. Includes "query". Ask
          the user to rephrase or name a nearby landmark.
        - "not_on_route": the place exists but the trip did not pass near it.
          Includes "closest_guess" (the best geocoded match). Tell the user the
          trip didn't go there and confirm the place.
    """
    candidates = geocode_place(place_name)
    if not candidates:
        return {"status": "not_found", "query": place_name}

    candidates_in_area = [c for c in candidates if inside_bbox(c["lat"], c["lon"], route_bbox(route))]
    if not candidates_in_area:
        return {"status": "not_on_route", "closest_guess": candidates[0]["name"]}
    
    if violation_mentioned:
        if violation_name:
            violation_name = fuzzy_match_violation(violation_name, trip_info, threshold=0.5)

        closest_point = closest_point_event(trip_info, violation_name, candidates, threshold_m=1000)
        
        if closest_point[0]:
            closest = closest_point[0]

            place = closest['place']["name"]
            event = closest.get('event')

            time_slot = get_time_slot_from_place_violation(closest, threshold_min)
            return {
                "status" : "success",
                "place" : place,
                "timeSlot" : time_slot
            }

    closest = closest_point_candidate(route, candidates_in_area)
    if not closest:
        return {"status": "not_on_route", "closest_guess": candidates[0]["name"]}

    time_slot = get_time_slot_from_point(closest["point"], threshold_min=threshold_min)
    return {
        "status": "success",
        "place": closest["place"]["name"],
        "timeSlot": time_slot
    }
route = get_trip_route(trips).get('route')


In [56]:
from datetime import datetime

# compares and finds the closest time 
def find_closest_violation_time(target_time, violation_list):
    target = datetime.fromisoformat(target_time.replace("Z", "+00:00"))

    closest = None
    smallest_diff = None

    for v in violation_list:
        time_str = v.get('event_timeUTC')
        dt = datetime.fromisoformat(time_str.replace("Z", "+00:00"))

        diff = abs(target - dt)

        if smallest_diff is None or diff < smallest_diff:
            smallest_diff = diff
            closest = time_str
            possible_violation = v

    # Reject if more than 1 hour away
    if smallest_diff is None or smallest_diff > timedelta(hours=1):
        return None

    return possible_violation

In [57]:
from openai import OpenAI
from rapidfuzz import process, fuzz
import numpy as np
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )


def get_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return [item.embedding for item in response.data]


def fuzzy_match_violation(violation, trip_info, threshold=0.5):
    event_count = trip_info.get("eventCount", {})
    violation_types = list(event_count.keys())

    if not violation_types:
        return None

    # Embed the user's violation
    violation_embedding = get_embeddings([violation])[0]

    # Embed all violation types in ONE API call
    candidate_embeddings = get_embeddings(violation_types)

    best_match = None
    best_score = -1

    for violation_type, candidate_embedding in zip(
        violation_types,
        candidate_embeddings
    ):
        score = cosine_similarity(
            violation_embedding,
            candidate_embedding
        )

        if score > best_score:
            best_score = score
            best_match = violation_type

    # Semantic match
    if best_score >= threshold:
        return best_match

    # Fuzzy/lexical fallback
    result = process.extractOne(
        violation,
        violation_types,
        scorer=fuzz.WRatio
    )

    if result and result[1] >= 85:
        return result[0]

    return None

print(fuzzy_match_violation("speed violation", trip_info))

Traffic-Speed-Violated


In [63]:
import time

start = time.time()

# your code here
from logger import debug_logger

debug_logger = debug_logger()
# we need to make sure that the violation argument should be fuzzy matched with 
# all the violation types that are present in the trip. If it is not present then 
# we need to ask the user to select a violation from the list of violations that are present in the trip.


def get_time_slot_info(trip_info, violation_name : str, violation_date_time : str = None, threshold_min : float = 1.5):
    Event_count = trip_info.get('eventCount', 0)
    debug_logger.info(Event_count)

    # if user hasn't mentioned any particular event, just mentioned that an event occurred then pull the list of the occurring events
    if violation_name is None:
        violation_count = Event_count.get('total')
    else:
        violation_count = Event_count.get(violation_name, 0)
    
    debug_logger.info(violation_count)

    if violation_count == 0:
        return {"status": f"no_violation", "message": f"{violation_name} did not occur in the trip."}

    if violation_count > 1:

        violations = trip_info.get("violations", [])

        possible_violation_references = (
            violations if violation_name is None
            else [v for v in violations if v.get("event_type") == violation_name])

        debug_logger.info(possible_violation_references)

        if violation_date_time is None:
            return {"status": "multiple_violations", "message": {"mssg" : f"multiple violations for {violation_name}", "options" : possible_violation_references}}
        # the planner node will extract the violation + time from the user selected option and call this function again with the violation_date_time argument.
        
        else:
            debug_logger.info(violation_date_time)
            violation_date_time = to_utc_string(
                time_str = violation_date_time
                )
            debug_logger.info(violation_date_time)
            debug_logger.info(violation_date_time, possible_violation_references)
            closest_violation = find_closest_violation_time(
                target_time = violation_date_time, 
                violation_list = possible_violation_references
            )

            debug_logger.info(closest_violation)

            if closest_violation is None:
                return {"status": "multiple_violations", 
                        "message": {"mssg" : f"there was no {violation_name} around {violation_date_time}. Select from the rest of the occurences of the {violation_name}", 
                                    "options" : possible_violation_references}}

            return {"status" : "success", "message" : closest_violation}

    if violation_count == 1:
        violation_date_time = trip_info.get('violations', [])[0].get('event_timeUTC')
        start_time = datetime.fromtimestamp(violation_date_time / 1000, tz=timezone.utc) - timedelta(minutes=threshold_min)
        end_time = datetime.fromtimestamp(violation_date_time / 1000, tz=timezone.utc) + timedelta(minutes=threshold_min)
        return {
            "status": "success",
            "violation": violations,
            "timeSlot": {
                "startTimeUTC": start_time.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
                "endTimeUTC": end_time.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            }
        }

trip_info = simplify_response(trips)

# when event_name + time, only event_name, event_name is mentioned but event occurred mentioned along with or without time

def get_time_slot_from_violation(trip_info, violation_name, violation_date_time, threshold_min=1.5):
    """
    Find the footage time slot for a violation/event that occurred in the selected trip.

    Purpose:
        Identifies which violation in the trip the user means and returns its
        time slot. The violation name is fuzzy-matched against the violation
        types present in the trip. If more than one violation could match,
        returns a list of options for the user to choose from.

    When to use:
        Use when the user says a violation or event occurred. Any of these
        cases counts, with or without a time:
        - Event named, no time:    "get the harsh braking clip"
        - Event named, with time:  "the overspeeding around 4:20 pm"
        - Event not named, no time:    "a violation occurred, get me the footage"
        - Event not named, with time:  "an event happened at 10 am, pull that video"
        Also use when the user picks one violation from options this tool
        returned earlier. Call it again with that option's name and time.

    When NOT to use:
        Do not use when the user identifies the moment by a LOCATION. This
        applies even if a violation is also mentioned:
        - "an event occurred near Silk Board"
        - "get me the DVR footage from when the truck was near Hosur"
        Use the location-based time-slot tool for these instead.

    Args:
        trip_info (dict): Simplified trip details for the selected trip. Must
            contain "eventCount" (event type → count, plus "total") and
            "violations" (list of violation dicts with "event_type" and
            "event_timeUTC").
        violation_name (str | None): Violation as the user phrased it, e.g.
            "speed violation". Pass None if the user only said that "an event"
            or "a violation" occurred without naming it.
        violation_date_time (str | None): When the violation happened, as an ISO
            8601 string, e.g. "2026-09-25T04:20:32.038Z". Pass None if the user
            gave no time.
        threshold_min (float): Minutes of footage on EACH side of the violation.
            Default 1.5, which gives a 3-minute clip.

    Returns:
        dict with a "status" key:
        - "success": a single violation was identified. Includes the violation
          and its time slot ("startTimeUTC", "endTimeUTC" as
          "YYYY-MM-DD HH:MM:SS.mmm").
        - "multiple_violations": several candidates matched and no time (or no
          close-enough time) was given. "message" contains "mssg" and "options"
          (list of candidate violations). Show the options to the user, then
          call this tool again with their choice.
        - "no_violation": the named violation did not occur in this trip. Tell
          the user and offer the violation types that did occur.
    """
    start = time.time()

    if violation_name is None:
        print(f"Seconds passed: {time.time() - start:.2f}")
        matched_violation = None
    else:
        matched_violation = fuzzy_match_violation(violation_name, trip_info)
        print(f"fuzzy: {matched_violation} {time.time() - start:.2f}")
        debug_logger.info(matched_violation)
    return get_time_slot_info(trip_info, matched_violation, violation_date_time, threshold_min)

print(get_time_slot_from_violation(trip_info, 'speed violation', '2026-09-25T04:20:32.038Z'))

2026-10-12 11:35:18,747 | debug_logger | INFO | RequestID: N/A | Traffic-Speed-Violated
2026-10-12 11:35:18,748 | debug_logger | INFO | RequestID: N/A | {'total': 15, 'Traffic-Speed-Violated': 3, 'Cornering': 0, 'Traffic-STOP-Sign-Violated': 0, 'Harsh-Braking': 0, 'Harsh-Acceleration': 0, 'Tail-Gating-Detected': 1, 'Lane-Drift-Found': 0, 'Distracted-Driving': 1, 'MaxSpeedExceeded': 0, 'Drowsy-Driving-Detected': 1, 'Forward-Collision-Warning': 5, 'Cellphone-Distracted-Driving': 0, 'Smoking-Distracted-Driving': 0, 'Drinking-Distracted-Driving': 0, 'Unbuckled-Seat-Belt': 0, 'Lizard-Eye-Distracted-Driving': 0, 'Roll-Over-Detected': 0, 'Texting-Distracted-Driving': 0, 'Traffic-Light-Violated': 0, 'Driver-Fatigue-Detected': 4}
2026-10-12 11:35:18,748 | debug_logger | INFO | RequestID: N/A | 3
2026-10-12 11:35:18,749 | debug_logger | INFO | RequestID: N/A | [{'event_index': 21, 'event_type': 'Traffic-Speed-Violated', ' ': '2026-09-25T04:20:32.038Z', 'event_location': {'latitude': 13.03989, 'l

fuzzy: Traffic-Speed-Violated 1.29


--- Logging error ---
Traceback (most recent call last):
  File "/Users/username/LISA/DVR Conversation/Agentic_DVR/backend/logger.py", line 145, in format
    formatted = super().format(record)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/logging/__init__.py", line 712, in format
    record.message = record.getMessage()
                     ~~~~~~~~~~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/logging/__init__.py", line 400, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/logging/handlers.py", line 79, in emit
    if self.shouldRollover(record):
       ~~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/logging/h

AttributeError: 'NoneType' object has no attribute 'replace'

In [ ]:
# when place is mentioned along with event
# when place is mentioned with time
# when violation is mentioned a



In [ ]:
trips_info = simplify_response(trips)
print(trips_info)
fuzzy_match_violation("harsh braking", trips_info)

{'tripId': 't_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z', 'startTime': '2026-09-25T09:46:04.333Z', 'endTime': '2026-09-25T10:44:23.807Z', 'duration': 3499, 'distance': None, 'averageSpeed': None, 'maxSpeed': None, 'asset_info': {'assetId': 'mithun-k245-fin', 'assetName': 'Mithun Mitac Fin', 'cameraType': 'RIDECAM_PLUS', 'dutyType': 'light'}, 'fleetId': 'lmfleet003', 'driverId': 'mithun', 'driverName': 'Mithun', 'eventCount': {'total': 15, 'Traffic-Speed-Violated': 3, 'Cornering': 0, 'Traffic-STOP-Sign-Violated': 0, 'Harsh-Braking': 0, 'Harsh-Acceleration': 0, 'Tail-Gating-Detected': 1, 'Lane-Drift-Found': 0, 'Distracted-Driving': 1, 'MaxSpeedExceeded': 0, 'Drowsy-Driving-Detected': 1, 'Forward-Collision-Warning': 5, 'Cellphone-Distracted-Driving': 0, 'Smoking-Distracted-Driving': 0, 'Drinking-Distracted-Driving': 0, 'Unbuckled-Seat-Belt': 0, 'Lizard-Eye-Distracted-Driving': 0, 'Roll-Over-Detected': 0, 'Texting-Distracted-Driving': 0, 'Traffic-Light-Violated': 0, 'Driver-Fatigue-Detected': 4}, '

'Harsh-Braking'

In [ ]:
# Scenarios
# • Place only ("near Hosur toll"): geocode the place, find the nearest route point, and use its time ± a buffer.
# • Violation type only ("the drowsy driving"): one event gives event time ± 1.5 min; several events let the user pick one.
# • Violation + time ("harsh braking around 3 PM"): take the event of that type closest to the time.
# • Violation + place ("speeding near the flyover"): take events of that type near the geocoded place.
# • Unnamed violation + time or place ("something happened at 3 PM"): take any violation closest to it; the user confirms if there are several.
# Additional scenarios to cover
# • Time only ("at 14:30"): use it directly after checking it falls inside the trip.
# • Time range ("10:00 to 10:05"): use as given, within trip bounds and clip / timelapse length limits.
# • Place + time ("near the depot around 9 AM"): pick the route point near the place closest to that time, since a trip can pass a place twice.
# • Two places ("from the warehouse to the highway"): time at the first place to time at the second, which suits a timelapse.
# • Part of the trip ("last 10 minutes", "whole trip"): derive from the trip's start and end.
# • Violation not in the trip: say so and list the violations that did occur.
# • Ambiguous or off-route place ("Bilaspur"): ask with options.
# • Repeat request ("same clip as last time"): reuse the window from previous upload requests.

In [ ]:
from fetch_trips.fetch_trips import fetch_trips_with_expiry
from dotenv import load_dotenv
load_dotenv()
url = "/v1/fleets/lmfleet003/drivers/mithun/trips/t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z"

print(fetch_trips_with_expiry(
    fleet_id='lmfleet003',
    client_id='lmdemotsp',
    start_time = "2026-09-24T00:00:00Z",
    end_time   = "2026-09-26T23:59:59Z",
    driver_ids='mithun'
))

2026-10-07 17:11:31,791 | debug_logger | DEBUG | RequestID: N/A | fetch_all_trips: skip=0 limit=500 got=11
2026-10-07 17:11:31,792 | debug_logger | INFO | RequestID: N/A | fetch_all_trips: total trips fetched 11
2026-10-07 17:11:31,793 | debug_logger | INFO | RequestID: N/A | fetch_trips_with_expiry: fetched 11 trips for fleet lmfleet003


[{'tripId': 't_NUQYNYFZXQI7DAAAZ4JIDYDS6A_Xymeb', 'driverId': 'mithun', 'driverName': 'Mithun', 'assetId': 'mithun-k245-fin', 'startTimeUTC': '2026-09-26T15:10:36.238Z', 'endTimeUTC': '2026-09-26T17:00:17.659Z', 'lastPinged': '2026-09-26T17:00:17.659Z', 'lastPingedLabel': 'Ended', 'dvr_status': 'available', 'dvr_until': '14 Oct 2026 03:10', 'dvr_days': 6.6, 'events': [{'type': 'Distracted-Driving', 'count': 4}, {'type': 'Drowsy-Driving-Detected', 'count': 1}, {'type': 'Forward-Collision-Warning', 'count': 1}, {'type': 'Cellphone-Distracted-Driving', 'count': 1}, {'type': 'Driver-Fatigue-Detected', 'count': 1}], 'totalEvents': 8}, {'tripId': 't_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z', 'driverId': 'mithun', 'driverName': 'Mithun', 'assetId': 'mithun-k245-fin', 'startTimeUTC': '2026-09-25T04:16:04.333Z', 'endTimeUTC': '2026-09-25T05:14:23.807Z', 'lastPinged': '2026-09-25T05:14:23.807Z', 'lastPingedLabel': 'Ended', 'dvr_status': 'available', 'dvr_until': '12 Oct 2026 16:16', 'dvr_days': 5.2, 'ev

In [ ]:
from upload_request.create_request import create_dvr_request
# params["driverId"],
#             "assetId": params["assetId"],
#             "tripId": params["tripId"],
#             "startTimeUTC": params["clipStart"],
#             "endTimeUTC": params["clipEnd"],
#             "dvrVideoType": params["videoFormat"],
#             "videoResolution": params["videoResolution"]


dvr_params = {
    "assetId": "mithun-k245-fin",
    "driverId": "mithun",
    "tripId": "t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z",
    "clipStart": "2026-09-25T04:17:32.038Z",
    "clipEnd": "2026-09-25T04:19:32.038Z",
    "dvrVideoType": "road",
    "videoResolution": "320x180"
}

print(create_dvr_request(
    fleet_id="lmfleet003",
    client_id="lmdemotsp",
    dvr_params= dvr_params
))

{'status': 'needs_confirmation', 'message': 'Please confirm the DVR clip request:\n- Request type: DVR clip\n- Driver: mithun\n- Asset: mithun-k245-fin\n- Start (UTC): 2026-09-25T04:17:32.038Z\n- End (UTC): 2026-09-25T04:19:32.038Z\n- Video format: road (options: road, driver, sideBySide, pictureInPicture, separate)\n- Resolution: 320x180 (options: 320x180, 640x360, 1280x720, 1920x1080)\nReply "confirm" to place the request, or tell me what to change.', 'params': {'type': 'clip', 'videoFormat': 'road', 'videoResolution': '320x180', 'assetId': 'mithun-k245-fin', 'driverId': 'mithun', 'tripId': 't_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z', 'clipStart': '2026-09-25T04:17:32.038Z', 'clipEnd': '2026-09-25T04:19:32.038Z', 'dvrVideoType': 'road'}}


In [ ]:
from time_slot.time_slot_agent import run_time_slot_agent

query = "Create a DVR request for a trip where there was a traffic speed violation near Bhuvanahalli"

run_time_slot_agent(
    user_query= query,
    trip=trips
)

2026-10-09 01:30:02,605 | debug_logger | DEBUG | RequestID: N/A | time_slot_agent: calling get_time_slot_from_place with {'place_name': 'Bhuvanahalli', 'violation_mentioned': True, 'violation_name': 'traffic speed violation', 'clip_type': 'dvr_clip', 'threshold_min': 1.5}
2026-10-09 01:30:06,663 | debug_logger | DEBUG | RequestID: N/A | fuzzy_match_violation: 'traffic speed violation' -> Traffic-Speed-Violated (0.738)
2026-10-09 01:30:06,664 | debug_logger | DEBUG | RequestID: N/A | time_slot_agent: get_time_slot_from_place returned {'status': 'success', 'basis': 'place_event', 'place': 'Bhuvanahalli - 560300, KA, India', 'violation': {'event_index': 155, 'event_type': 'Traffic-Speed-Violated', 'event_timeUTC': '2026-09-25T04:55:21.034Z', 'event_location': {'latitude': 13.22367, 'longitude': 77.68449}}, 'distM': 731, 'timeSlot': {'startTimeUTC': '2026-09-25 04:53:51.034', 'endTimeUTC': '2026-09-25 04:56:51.034'}}


{'status': 'success',
 'time_slot': {'startTimeUTC': '2026-09-25 04:53:51.034',
  'endTimeUTC': '2026-09-25 04:56:51.034'},
 'thinking': 'Place: Bhuvanahalli - 560300, KA, India\nEvent: Traffic-Speed-Violated at 04:55:21 AM\nDVR clip: 04:53:51 AM to 04:56:51 AM (UTC)',
 'message': None,
 'options': [],
 'tool_result': {'status': 'success',
  'basis': 'place_event',
  'place': 'Bhuvanahalli - 560300, KA, India',
  'violation': {'event_index': 155,
   'event_type': 'Traffic-Speed-Violated',
   'event_timeUTC': '2026-09-25T04:55:21.034Z',
   'event_location': {'latitude': 13.22367, 'longitude': 77.68449}},
  'distM': 731,
  'timeSlot': {'startTimeUTC': '2026-09-25 04:53:51.034',
   'endTimeUTC': '2026-09-25 04:56:51.034'}}}

In [66]:
def get_drivers(fleet_id: str, client_id: str, driver_name: str | None = None, page_size: int = 500):
    """
    Pages through /fleets/{fleetId}/drivers/list and returns
    [{'driverName': ..., 'driverId': ...}]. If driver_name is given, only
    drivers whose name contains it (case-insensitive) are returned.
    """
    driver_url = f"/v2/fleets/{fleet_id}/drivers/list"

    drivers_list = []
    skip = 0
    while True:
        body = auth_manager.make_api_request(
            client_id=client_id,
            endpoint=driver_url,
            params={"limit": page_size, "skip": skip},
        )
        rows = body.get("rows", [])
        drivers_list.extend(
            {"driverName": row.get("driverName"), "driverId": row.get("driverId")}
            for row in rows
        )
        skip += len(rows)
        if not rows or len(rows) < page_size or skip >= body.get("totalCount", 0):
            break

    if driver_name:
        drivers_list = [
            d for d in drivers_list
            if driver_name.lower() in (d["driverName"] or "").lower()
        ]
    return drivers_list


drivers_list = get_drivers(fleet_id="lmfleet003", client_id="lmdemotsp", driver_name="mithun")
drivers_list

[{'driverName': 'Mithun', 'driverId': 'mithun'},
 {'driverName': 'Mithun Uliyar', 'driverId': 'mithunUliyar'}]

In [67]:
def get_asset(fleet_id: str, client_id: str, asset_name: str | None = None):
    
    asset_url = f"/v2/fleets/{fleet_id}/list-fleet-assets"

    body = auth_manager.make_api_request(
        client_id=client_id,
        endpoint=asset_url,
    )
    asset_list = [
        {"assetName": row.get("assetName"), "assetId": row.get("assetId")}
        for row in body.get("rows", [])
    ]

    if asset_name:
        asset_list = [
            a for a in asset_list
            if asset_name.lower() in (a["assetName"] or "").lower()
        ]
    return asset_list


asset_list = get_asset(fleet_id="lmfleet003", client_id="lmdemotsp")
asset_list

[{'assetName': None, 'assetId': '350892690000258'},
 {'assetName': None, 'assetId': '812345678987654'},
 {'assetName': "Anuradha's mitac", 'assetId': '861778062355465'},
 {'assetName': "Anuradha's jimmy", 'assetId': '862798052772726'},
 {'assetName': 'Geetika001', 'assetId': '864281040862220'},
 {'assetName': 'Kruthika-Honda', 'assetId': '864281040864283'},
 {'assetName': 'Soumik Hyryder', 'assetId': '864281042302902'},
 {'assetName': 'Kanika K245', 'assetId': '864281042310145'},
 {'assetName': 'Mithun-K220', 'assetId': '864281043957548'},
 {'assetName': 'Krishna_fin2', 'assetId': '869409069763244'},
 {'assetName': 'Ashish GrandVitara', 'assetId': 'Ashish-GrandVitara'},
 {'assetName': 'Lalit', 'assetId': 'GJ05RK1495'},
 {'assetName': 'Hyundai Aura', 'assetId': 'Hyndai-Aura'},
 {'assetName': 'KrishnaSX4 JC400', 'assetId': 'KrishnaSX4-JC400'},
 {'assetName': 'Mithun JC400', 'assetId': 'mithun-jc400'},
 {'assetName': 'Mithun Mitac Fin', 'assetId': 'mithun-k245-fin'},
 {'assetName': 'Raghu

In [68]:
import os
from datetime import datetime, timezone
from typing import Literal
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

from utils.auth_util import AuthManager
from state import AgentState, DateTimeRange, AssetInfo, DriverInfo
load_dotenv()

auth_manager = AuthManager()

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
    api_key=os.environ.get("OPENAI_API_KEY")
)

EXTRACT_PROMPT = """
You are an information extraction system for a fleet management DVR assistant.
Users ask for DVR clips or timelapses of a vehicle's camera footage.

Your task is to extract structured information from the user's query.
A query usually contains only some of the fields below. Extract only the fields that are
present in the query and leave every other field as None.

Current date/time (UTC): {now}

Extract the following fields:

1. driver_name
   * The driver's name as written in the query, e.g. "John", "Sunil Kumar".
   * Return None if no driver is mentioned.

2. asset_name
   * The vehicle / asset the user refers to: its name, number or plate,
     e.g. "Truck 12", "Bus-042", "KA01AB1234".
   * Copy it as written. Return None if no vehicle is mentioned.

3. trip_id
   * A trip identifier, copied exactly character for character,
     e.g. "t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z".
   * Trip IDs do not follow one fixed pattern. They are usually a single long token of
     letters, digits, underscores or hyphens, and often follow words like "trip" or
     "trip id", or appear inside "[Trip: ...]".
   * Never shorten, re-case or correct it, and never treat a driver or asset name as a trip ID.
   * Return None if no trip ID is present.

4. event_name and violation_mentioned
   * violation_mentioned: true when the user says a safety event / violation occurred, whether
     or not they name it, e.g. "the harsh braking", "a violation happened at 10am",
     "something happened near Hosur". Otherwise false.
   * event_name: the specific event the user names, in the user's own words,
     e.g. "harsh braking", "overspeeding", "using the phone while driving".
     Do not map it to a standard event type, that is checked later.
   * If the user only refers to an event in general ("an event", "a violation", "an incident",
     "something happened"), set violation_mentioned = true and event_name = None.
   * Return event_name None if no specific event is named.

5. location
   * The place the footage should be from: an address, street, area, city, landmark or
     business, e.g. "the Walmart on 5th Street", "Downtown Chicago", "the toll plaza on I-90".
   * Copy the place as written. Return None if no place is mentioned.

6. clip_type
   * "dvr_clip" when the user wants a video clip, footage, recording or DVR.
   * "timelapse" when the user asks for a timelapse.
   * Return None if the query doesn't say which one.

7. request_duration
   * The length of footage the user asks for, in minutes,
     e.g. "a 2 minute clip" -> 2, "30 seconds of video" -> 0.5, "one and a half minutes" -> 1.5.
   * Return None if no length is given. Don't derive it from the start and end times.

8. video_format
   * The camera view the user wants, exactly one of:
       road              - road-facing / front / outward camera, dashcam view
       driver            - driver-facing / cabin / inward camera
       sideBySide        - both views next to each other
       pictureInPicture  - one view shown inside the other
       separate          - both views as separate videos
   * Return the exact value above. Return None if no view is mentioned.

9. video_resolution
   * Exactly one of: 320x180, 640x360, 1280x720, 1920x1080.
   * Map common names: "180p" -> 320x180, "360p" -> 640x360, "720p" / "HD" -> 1280x720,
     "1080p" / "Full HD" -> 1920x1080.
   * Return None if no resolution is mentioned.

10. available_duration
   * How much recorded footage the user says is available, as written, e.g. "only 5 minutes".
   * Return None unless the user states it explicitly.

11. date_time (startTimeUTC, endTimeUTC)
   * The time window the footage / trip should be from, in UTC, formatted as
     YYYY-MM-DDTHH:MM:SS. Times in the query are UTC unless the user gives a timezone,
     in which case convert them to UTC.
   * Explicit range ("5th June between 9 and 10 AM") -> its start and end.
   * Date with no time ("5th June") -> 00:00:00 to 23:59:59 of that date.
   * Date AND a single clock time with no window ("5th June 9AM") -> 1 hour before to
     1 hour after that time (08:00:00 to 10:00:00).
   * Clock time with no date ("at 9AM") -> apply it to today.
   * Date without a year ("16th June") -> the current year, or the previous year if that
     date would be in the future.
   * Relative references are resolved against the current date/time above:
     "today" / "yesterday" -> that whole day, "last 3 days" -> now minus 3 days to now,
     "last week" -> now minus 7 days to now, "last month" -> now minus 30 days to now.
   * If only one bound is given ("after 5PM yesterday"), the other bound is the start or
     end of that same day.
   * Return None for both if the query has no date or time information.

   Time of day (when the query names a part of the day instead of a clock time, e.g.
   "in the morning", "after sunrise", "during rush hour", "late at night"):
   * Use your general understanding of what time of day the phrase conventionally refers
     to, the way a person would in everyday conversation. Reference scale (24-hour clock):
       midnight ~ 00:00   early morning ~ 04:00-06:00   sunrise/dawn ~ 06:00
       morning ~ 06:00-12:00   noon ~ 12:00   afternoon ~ 12:00-17:00
       evening ~ 17:00-20:00   sunset/dusk ~ 18:00-19:00   night ~ 20:00-24:00
     Interpret other phrases (rush hour, lunch hour, post-midnight, before dawn...) by
     placing them on the same scale.
   * Build a window of roughly 1-4 hours around your interpretation, unless the wording
     clearly means a single moment ("right after sunrise").
   * "before <phrase>" -> start of the day to the start of the phrase's window.
     "after <phrase>"  -> end of the phrase's window to the end of the day, unless another
     bound is given. "during <phrase>" or no preposition -> the window itself.
   * Apply the window to the date that is otherwise resolved, or today if there is none.
   * If an explicit clock time is also given ("morning, around 9AM"), use the clock time.
   * If the phrase is ambiguous, don't guess: treat it as if no time of day was given.

12. event_time
   * The exact moment the user points to with a single clock time, in UTC, formatted as
     YYYY-MM-DDTHH:MM:SSZ, e.g. "the overspeeding at 4:20 PM on 5th June" -> "2026-06-05T16:20:00Z".
   * Resolve its date the same way as date_time (today if no date is given).
   * It is set alongside date_time, not instead of it: "5th June 9AM" gives date_time
     08:00:00 to 10:00:00 AND event_time "2026-06-05T09:00:00Z".
   * Return None if the query has only a date, a range or a part of the day, and no single clock time.

13. limit_to_latest
   * Set when the user asks for the latest / most recent trips instead of a time window:
       "the latest trip", "the most recent trip", "last trip"        -> 1
       "the last 3 trips", "most recent 10 trips"                    -> that number
       "latest trips", "recent trips" (plural, no count)             -> 50
   * "last" + a unit of time ("last week", "last 5 days", "past month") is a time window,
     not a count: use date_time for it and leave limit_to_latest None.
     "last" + trip/trips or a bare count ("last 5 trips") is a count.
   * limit_to_latest and date_time are never both set. If the query also gives a date, a range,
     a relative duration or a part of the day, use date_time and leave limit_to_latest None.
   * Return None if the query doesn't ask for the latest trips.

Clarifying questions
* The query may be followed by a clarifying question asked to the user and the option they
  selected. Use the selected option, exactly as written, as the value of the field named in
  its purpose (e.g. purpose "driver_name" -> driver_name = the selected option), and keep
  extracting every other field from the original query.
* If the purpose names more than one field, put the answer in the field it belongs to: a name
  from the "drivers" options -> driver_name, from the "assets" options -> asset_name, a date or
  time -> date_time. The answer may be typed by the user instead of picked from the options.

General Rules
1. Extract only information present in the query (or the user's answer to a clarifying
   question). Never invent values.
2. Return None for every field that is not present.
3. Copy names, IDs and places as written, without correcting their spelling.
4. video_format, video_resolution and clip_type must be one of their listed values or None.
"""

class Extract(BaseModel):

    driver_name : str | None = None

    asset_name : str | None = None

    date_time : DateTimeRange

    event_name : str | None = None

    violation_mentioned : bool = False

    event_time : str | None = None

    clip_type : Literal["dvr_clip", "timelapse"] | None = None

    request_duration : float | None = None

    video_format : str | None = None

    available_duration : str | None = None

    video_resolution : str | None = None

    location : str | None = None

    trip_id : str | None = None

    limit_to_latest : int | None = None


EVENT_COUNTS = {'total': 15, 'Traffic-Speed-Violated': 3, 'Cornering': 0, 'Traffic-STOP-Sign-Violated': 0, 'Harsh-Braking': 0, 'Harsh-Acceleration': 0, 'Tail-Gating-Detected': 1, 'Lane-Drift-Found': 0, 'Distracted-Driving': 1, 'MaxSpeedExceeded': 0, 'Drowsy-Driving-Detected': 1, 'Forward-Collision-Warning': 5, 'Cellphone-Distracted-Driving': 0, 'Smoking-Distracted-Driving': 0, 'Drinking-Distracted-Driving': 0, 'Unbuckled-Seat-Belt': 0, 'Lizard-Eye-Distracted-Driving': 0, 'Roll-Over-Detected': 0, 'Texting-Distracted-Driving': 0, 'Traffic-Light-Violated': 0, 'Driver-Fatigue-Detected': 4}

EVENT_TYPES = [e for e in EVENT_COUNTS if e != "total"]

MAX_CLIP_MIN = {"dvr_clip": 3, "timelapse": 60}     # longest clip allowed for each clip_type

MAP_EVENT_PROMPT = """
Map the safety event the user mentioned to one of the supported event types.

Supported event types:
{event_types}

* Pick the event type that means the same as the user's words, including synonyms and
  paraphrases, e.g. "tailgating" -> Tail-Gating-Detected, "on the phone" -> Cellphone-Distracted-Driving,
  "ran a red light" -> Traffic-Light-Violated.
* If more than one fits, pick the most specific one.
* Return the event type exactly as written in the list above.
* Return None if none of them match. Don't treat general words like "accident" as an event.

User's event: {event_name}
"""


class EventMatch(BaseModel):

    event_type : Literal[tuple(EVENT_TYPES)] | None = None


def map_event(event_name: str):
    """Maps the event in the user's words to one of EVENT_TYPES, None if none of them match."""
    prompt = MAP_EVENT_PROMPT.format(event_types="\n".join(EVENT_TYPES), event_name=event_name)
    return llm.with_structured_output(EventMatch).invoke(prompt).event_type


def resolve_one(matches: list[dict], searched: str, kind: str, name_key: str, id_key: str):
    """
    Returns (id, None) when exactly one match is found, otherwise
    (None, response) where response is what the node should return.
    """
    # an exact name match wins, so picking "John" from the options
    # doesn't match "John Smith" again on the next pass
    exact = [m for m in matches if (m[name_key] or "").lower() == searched.lower()]
    if len(exact) == 1:
        return exact[0][id_key], None

    if not matches:
        return None, {
            "next_node" : "END",
            "reason" : f"No Matching {kind.title()}",
            "message" : f"No {kind} found matching '{searched}'. Please check the name and try again.",
            "options" : []
        }

    if len(matches) > 1:
        names = [m[name_key] for m in matches]
        question = f"Multiple {kind}s match '{searched}'. Which one did you mean?"
        return None, {
            "next_node" : "END",
            "reason" : f"Multiple Matching {kind.title()}s",
            "message" : f"{question} Please Select an option",
            "options" : names,
            "cross_question" : {
                "node" : "extract",
                "question" : question,
                "options" : names,
                "purpose" : f"{kind}_name",
                "user_feedback" : None
            }
        }

    return matches[0][id_key], None


def extract(state):

    query = state["query"]

    cross_question = state.get("cross_question")

    cross_question_feedback = ""

    # if one of the node asked a cross_question
    # node (asked a question) --> end_orchestrator (ends the flow) [user selects an option or writes something else] 
    if cross_question:
        cq_node = cross_question["node"]
        question = cross_question["question"]
        options = cross_question["options"]
        purpose = cross_question["purpose"] # to fill information in which field of the state
        user_response = cross_question["user_feedback"]

        # no option picked: the user typed a message instead, which may answer the question
        # or ask for something else, the planner decides which
        if user_response is None:
            cross_question_feedback = f"""
            Node {cq_node} asked a clarifying question {question} from the user
            for the purpose {purpose}. The options provided to the user were {options}.
            The user didn't pick an option and replied with the query below instead, which may
            answer the question or ask for something else"""

        else:
            cross_question_feedback = f"""
            Node {cq_node} asked a clarifying question {question} from the user
            for the purpose {purpose}. The options provided to the user were {options}
            and the user selected the option {user_response}"""


    final_prompt = f"""
    {EXTRACT_PROMPT.format(now=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"))}
    user's query : {query}
    {cross_question_feedback}"""

    structured_llm = llm.with_structured_output(Extract)

    output = structured_llm.invoke(final_prompt)

    event_name = output.event_name
    location = output.location
    clip_type = output.clip_type or state.get("clip_type")
    video_format = output.video_format or state.get("video_format")
    video_resolution = output.video_resolution or state.get("video_resolution")
    request_duration = output.request_duration
    event_time = output.event_time
    violation_mentioned = output.violation_mentioned or bool(event_name)


    # compare if there has been a change in the primary filters
    # (driver, time, date)
    prev_driver = state.get("driver") or DriverInfo()
    prev_asset = state.get("asset") or AssetInfo()
    prev_date_time = state.get("date_time")
    prev_limit = state.get("limit_to_latest")

    driver_name = output.driver_name or prev_driver.driver_name
    asset_name = output.asset_name or prev_asset.asset_name
    trip_id = output.trip_id or state.get("trip_id")

    # a time window and limit_to_latest are never both set, the one in this message wins
    if output.date_time is not None and output.date_time.startTimeUTC is not None:
        date_time, limit_to_latest = output.date_time, None
    elif output.limit_to_latest:
        date_time, limit_to_latest = None, output.limit_to_latest
    else:
        date_time, limit_to_latest = prev_date_time, prev_limit

    # a trip_id is one trip on its own; otherwise fetch_trips needs a time window or limit_to_latest,
    # and a driver or asset so it doesn't return too many trips to pick from
    has_window = date_time is not None and date_time.startTimeUTC is not None
    has_trips_range = has_window or limit_to_latest
    has_driver_or_asset = driver_name or asset_name
    if not trip_id and not (has_trips_range and has_driver_or_asset):
        missing, purposes, options = [], [], []
        if not has_trips_range:
            missing.append("a date or date range (or ask for the latest trips)")
            purposes.append("date_time")
        if not has_driver_or_asset:
            missing.append("a driver or an asset")
            purposes.append("driver_name or asset_name")
            options = {
                "drivers" : [d["driverName"] for d in get_drivers(fleet_id = state['fleet_id'], client_id = state['client_id']) if d["driverName"]],
                "assets" : [a["assetName"] for a in get_asset(fleet_id = state['fleet_id'], client_id = state['client_id']) if a["assetName"]]
            }
        question = f"Please provide {' and '.join(missing)}, or a trip ID."
        return {
            "next_node" : "END",
            "reason" : "Missing Params",
            "message" : question,
            "options" : options,
            "cross_question" : {
                "node" : "extract",
                "question" : question,
                "options" : options,
                "purpose" : ", ".join(purposes),
                "user_feedback" : None
            }
        }

    # map the event in the user's words to a supported event type
    event_type = None
    if event_name:
        event_type = map_event(event_name)
        if event_type is None:
            question = f"'{event_name}' doesn't match any supported event. Which event did you mean?"
            return {
                "next_node" : "END",
                "reason" : "No Matching Event",
                "message" : f"{question} Please Select an option",
                "options" : EVENT_TYPES,
                "cross_question" : {
                    "node" : "extract",
                    "question" : question,
                    "options" : EVENT_TYPES,
                    "purpose" : "event_name",
                    "user_feedback" : None
                }
            }

    # only look up what the user actually mentioned, a None name would match everything
    # reuse the ID found earlier when the driver didn't change
    driver_id = prev_driver.driver_id if driver_name == prev_driver.driver_name else None
    if driver_name and driver_id is None:
        drivers = get_drivers(
            fleet_id = state['fleet_id'], client_id = state['client_id'], driver_name = driver_name
        )
        driver_id, response = resolve_one(drivers, driver_name, "driver", "driverName", "driverId")
        if response:
            return response

    asset_id = prev_asset.asset_id if asset_name == prev_asset.asset_name else None
    if asset_name and asset_id is None:
        assets = get_asset(
            fleet_id = state['fleet_id'], client_id = state['client_id'], asset_name = asset_name
        )
        asset_id, response = resolve_one(assets, asset_name, "asset", "assetName", "assetId")
        if response:
            return response

    # the time slot tools take half the clip length on each side of the moment
    threshold_min = None
    if request_duration:
        threshold_min = min(request_duration, MAX_CLIP_MIN[clip_type or "dvr_clip"]) / 2

    # the planner compares these with the filters after this message to decide if the user changed them
    previous_filters = {
        "driver_name" : prev_driver.driver_name,
        "driver_id" : prev_driver.driver_id,
        "asset_name" : prev_asset.asset_name,
        "asset_id" : prev_asset.asset_id,
        "start_time" : prev_date_time.startTimeUTC if prev_date_time else None,
        "end_time" : prev_date_time.endTimeUTC if prev_date_time else None,
        "limit_to_latest" : prev_limit,
        "trip_id" : state.get("trip_id")
    }

    history = list(state.get("history") or [])
    history.append(HumanMessage(query))

    result = {
        "driver" : DriverInfo(driver_id = driver_id, driver_name=driver_name),
        "asset" : AssetInfo(asset_id = asset_id, asset_name= asset_name),
        "date_time" : date_time,
        "location" : location,
        "event_mentioned" : event_name,
        "event_type" : event_type,
        "violation_mentioned" : violation_mentioned,
        "event_time" : event_time,
        "trip_id" : trip_id,
        "limit_to_latest" : limit_to_latest,
        "clip_type" : clip_type,
        "video_format" : video_format,
        "video_resolution" : video_resolution,
        "request_duration" : request_duration,
        "threshold_min" : threshold_min,
        "history" : history,
        "previous_filters" : previous_filters,
        "new_message" : True
    }

    # only clear a cross question this node asked, one asked by another
    # node stays in the state so that node can read the user's answer
    if cross_question and cross_question["node"] == "extract":
        result["cross_question"] = None

    return result



In [77]:
state = {
    'query' : "Show the latest trips for driver Mithun on 25th september 2026 around 7pm, there was speed violaiton. I need a dvr clip for the same with the road front ",
    'fleet_id' : "lmfleet003",
    "client_id" : 'lmdemotsp',
    "driver" : DriverInfo(driver_name='Soumik'),
    "date_time" : DateTimeRange(startTimeUTC="2026-06-25T18:00:00")
}

extract(state)

{'driver': DriverInfo(driver_id='mithun', driver_name='Mithun'),
 'asset': AssetInfo(asset_id=None, asset_name=None),
 'date_time': DateTimeRange(startTimeUTC='2026-09-25T18:00:00', endTimeUTC='2026-09-25T20:00:00'),
 'location': None,
 'event_mentioned': 'speed violaiton',
 'event_type': 'Traffic-Speed-Violated',
 'violation_mentioned': True,
 'event_time': '2026-09-25T19:00:00Z',
 'trip_id': None,
 'limit_to_latest': None,
 'clip_type': 'dvr_clip',
 'video_format': 'road',
 'video_resolution': None,
 'request_duration': None,
 'threshold_min': None,
 'history': [HumanMessage(content='Show the latest trips for driver Mithun on 25th september 2026 around 7pm, there was speed violaiton. I need a dvr clip for the same with the road front ', additional_kwargs={}, response_metadata={})],
 'previous_filters': {'driver_name': 'Soumik',
  'driver_id': None,
  'asset_name': None,
  'asset_id': None,
  'start_time': '2026-06-25T18:00:00',
  'end_time': None,
  'limit_to_latest': None,
  'trip_i

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel

from state import AgentState

Action = Literal["fetch_trip", "dvr", "trip_qna", "prev_upload_requests", "none"]

# nodes each action runs, when the trip is already known
ACTION_PLANS = {
    "fetch_trip" : ["fetch_trip"],
    "dvr" : ["time_slot", "dvr"],
    "trip_qna" : ["trip_qna"],
    "prev_upload_requests" : ["prev_upload_requests"],
    "none" : []
}

TRIP_NODES = {"time_slot", "dvr", "trip_qna", "prev_upload_requests"}     # need one trip to work on

MAX_HISTORY_TURNS = 10

NOTHING_TO_DO_MESSAGE = ("I can show trips, answer questions about a trip, list its previous upload requests, "
                         "or request a DVR clip or timelapse. What would you like to do?")

PLANNER_PROMPT = """
You plan how a fleet management DVR assistant handles the user's latest message.

Actions:
- fetch_trip: show / list / find trips matching filters (driver, asset, date, events, latest N trips).
- dvr: get a DVR clip or a timelapse of a trip's camera footage.
- trip_qna: answer a question about one trip (distance, speed, events, driver, start / end time...).
- prev_upload_requests: list the DVR clip / timelapse requests already made for a trip.
- none: the message needs none of these (a greeting, off-topic, or too unclear to act on).

Ongoing request:
- Pending plan (nodes still to run): {plan}
- Intent: {intent}
- Waiting for the user's answer to: {pending_question}

Trip filters before this message: {previous_filters}
Trip filters after this message:  {current_filters}
(Filters the message doesn't mention carry over from the earlier messages.)

Decide filters_changed, by comparing the filters before and after this message together with
the message and the conversation:
- true when the user now wants DIFFERENT trips: another driver, another asset, or a date / time
  range that points to other trips than before ("actually Ramesh's trip", "no, the 6th June
  trips", "show truck 12 instead").
- false when the filters are the same, or the message only narrows down or refines the same
  trips: e.g. giving the time of the moment within the selected trip ("around 4 PM") or
  answering a question about them.
- false when there were no filters before this message.

Decide query_type. A new message is not by itself a new query, it is often a follow-up:
- cross_query: the message is a follow-up to the ongoing request. It answers the pending question,
  picks an option, gives or changes a parameter of it (date, driver, asset, place, event, clip
  length, video format...), or confirms it. Set action to the ongoing request's action.
- new_query: the user has moved on from the ongoing request and asks for something else: a
  different action (e.g. a question about the trip in the middle of a DVR request), or the same
  action for an unrelated request. Set action to what the new request needs. Also new_query when
  there is no ongoing request.

user_intent: one sentence stating the requested action, its filters and the objective, built from
the message, the filters and the conversation. For a cross_query, state the updated intent of the
whole request, not just the follow-up.
e.g. "Create a 2 minute DVR clip of John's trip on 2026-06-05 at the harsh braking near Hosur."

Other extracted parameters: {params}

Conversation history:
{history}

User's message: {query}
"""


class PlannerDecision(BaseModel):

    filters_changed : bool

    query_type : Literal["new_query", "cross_query"]

    action : Action

    user_intent : str


def current_filters(state: AgentState):
    """The trip filters after the latest message, in the same shape as previous_filters."""
    driver, asset, date_time = state.get("driver"), state.get("asset"), state.get("date_time")
    return {
        "driver_name" : driver.driver_name if driver else None,
        "driver_id" : driver.driver_id if driver else None,
        "asset_name" : asset.asset_name if asset else None,
        "asset_id" : asset.asset_id if asset else None,
        "start_time" : date_time.startTimeUTC if date_time else None,
        "end_time" : date_time.endTimeUTC if date_time else None,
        "limit_to_latest" : state.get("limit_to_latest"),
        "trip_id" : state.get("trip_id")
    }


def other_params(state: AgentState):
    """The rest of what the Extract node put in the state, for the planner prompt."""
    return {
        "event" : state.get("event_type") or state.get("event_mentioned"),
        "event_time" : state.get("event_time"),
        "location" : state.get("location"),
        "clip_type" : state.get("clip_type"),
        "request_duration" : state.get("request_duration")
    }


def format_history(history):
    """Last MAX_HISTORY_TURNS turns as 'role: content' lines."""
    lines = []
    for turn in (history or [])[-MAX_HISTORY_TURNS:]:
        if isinstance(turn, dict):
            lines.append(f"{turn.get('role')}: {turn.get('content')}")
        else:
            lines.append(f"{turn.type}: {turn.content}")
    return "\n".join(lines) or "(none)"


def ensure_trip(plan, trip_ready):
    """Puts fetch_trip first when the first node needs a trip whose details aren't loaded yet."""
    if plan and plan[0] in TRIP_NODES and not trip_ready:
        return ["fetch_trip", *plan]
    return plan


def planner(state):
    """
    1. Back from the end orchestrator with no new user message: continues the pending plan.
    2. New user message: the LLM compares the filters before and after it to decide if the user
    changed them, and decides if the message is a follow-up (cross_query, the pending plan carries on)
    3. the user moved on to something else (new_query, a whole new plan).
    """

    # if there is an ongoing plan
    plan = list(state.get("plan") or [])

    # a node finished and steps remain, nothing new from the user to plan for
    if not state.get("new_message"):
        return {
            "plan" : plan,
            "planner_query_type" : "ongoing_execution",
            "next_node" : plan[0] if plan else "end_orchestrator"
        }

    previous = state.get("previous_filters") or {}
    current = current_filters(state)
    cross_question = state.get("cross_question")
    prompt = PLANNER_PROMPT.format(
        plan = plan or "(none)",
        intent = state.get("user_intent") or "(none)",
        pending_question = cross_question["question"] if cross_question else "(none)",
        previous_filters = json.dumps(previous, default=str),
        current_filters = json.dumps(current, default=str),
        params = json.dumps(other_params(state), default=str),
        history = format_history(state.get("history")),
        query = state["query"]
    )
    decision = llm.with_structured_output(PlannerDecision).invoke(prompt)

    query_type = decision.query_type if plan else "new_query"
    filters_changed = decision.filters_changed and any(previous.values())

    result = {
        "planner_query_type" : query_type,
        "user_intent" : decision.user_intent,
        "filters_changed" : filters_changed,
        "new_message" : False       # planned for this message, the loop back from the end orchestrator only continues
    }

    trip_id = state.get("trip_id")
    if filters_changed:
        # the trips and the selected trip were for the old filters, and so was the conversation
        if trip_id == previous.get("trip_id"):
            trip_id = None              # unless this message named a new trip
        result.update(trips = [], trip_id = trip_id, selected_trip = None, time_slot = None, dvr_params = None,
                      history = (state.get("history") or [])[-1:])

    # fetch_trip loads the trip details, also for a trip ID the user typed
    selected_trip = None if filters_changed else state.get("selected_trip")
    trip_ready = bool(trip_id and selected_trip and selected_trip.get("tripId") == trip_id)

    # completed nodes already removed themselves, so continuing the plan never repeats them;
    # changed filters rebuild it, as the nodes already done worked on the old trips
    if query_type == "cross_query" and not filters_changed:
        new_plan = plan
    else:
        new_plan = list(ACTION_PLANS[decision.action])
    new_plan = ensure_trip(new_plan, trip_ready)

    result.update(plan = new_plan, next_node = new_plan[0] if new_plan else "end_orchestrator")

    # the user moved on, a question a node asked about the old request no longer needs an answer
    if query_type == "new_query" and cross_question and cross_question["node"] != "extract":
        result["cross_question"] = None
    if not new_plan:
        result.update(reason = "Nothing To Do", message = NOTHING_TO_DO_MESSAGE)
    return result


def route_after_planner(state: AgentState):
    """To the first pending node, or to the end orchestrator when nothing is pending."""
    plan = state.get("plan") or []
    return plan[0] if plan else "end_orchestrator"

In [ ]:
state = {
    "plan" : ["fetch_trips", "time_slot", "dvr"]
    "new_message" : False,
    "previous_filters" : {'driver_name': 'Soumik',
                          'driver_id': None,
                          'asset_name': None,
                          'asset_id': None,
                          'start_time': '2026-06-25T18:00:00',
                          'end_time': None,
                          'limit_to_latest': None,
                          'trip_id': None},
    "cross_question" : "",
    "trip_id" : "t_2LNEWUFYS4I7DAAAZ4JIDYDS6A_8CT1z",
}

planner(state)

{'plan': [],
 'planner_query_type': 'ongoing_execution',
 'next_node': 'end_orchestrator'}

In [ ]:
# node helpers
import json
from functools import wraps

import httpx
import openai

from logger import debug_logger
from state import AgentState

logger = debug_logger()


def is_recoverable(error: Exception):
    """Network problems, timeouts, rate limits and server errors may work on a retry; the rest won't."""
    if isinstance(error, httpx.HTTPStatusError):
        return error.response.status_code == 429 or error.response.status_code >= 500
    return isinstance(error, (httpx.TransportError, openai.APIConnectionError,
                              openai.RateLimitError, openai.InternalServerError))


def graph_node(name):
    """Runs the node, turning an exception into an error result for the end orchestrator."""
    def wrap(fn):
        @wraps(fn)
        def node(state: AgentState):
            try:
                return fn(state)
            except Exception as e:
                logger.error(f"{name} failed: {e}", exc_info=True)
                return {"last_result": {"node": name, "status": "error", "error": str(e),
                                        "recoverable": is_recoverable(e)}}
        return node
    return wrap


def own_answer(state: AgentState, node):
    """The user's answer to the question this node asked: the option picked, or the message typed. None if no question."""
    question = state.get("cross_question")
    if not question or question["node"] != node:
        return None
    answer = question.get("user_feedback")
    if answer is None:
        return state["query"]
    return answer if isinstance(answer, str) else json.dumps(answer, default=str)


def clear_own_question(state: AgentState, node):
    question = state.get("cross_question")
    return {"cross_question": None} if question and question["node"] == node else {}


def done(state: AgentState, node, message=None, data=None, **update):
    """The node finished: it removes itself from the plan and hands its result to the end orchestrator."""
    plan = list(state.get("plan") or [])
    if plan and plan[0] == node:
        plan = plan[1:]
    return {
        "plan" : plan,
        "last_result" : {"node": node, "status": "done", "message": message, "data": data},
        **clear_own_question(state, node),
        **update
    }


def ask(state: AgentState, node, question, options=None, purpose="", **update):
    """The node needs the user to pick / answer something. It stays in the plan and runs again with the answer."""
    cross_question = {"node": node, "question": question, "options": options or [],
                      "purpose": purpose, "user_feedback": None}
    return {
        "cross_question" : cross_question,
        "last_result" : {"node": node, "status": "needs_input", "message": question,
                         "options": options or [], "cross_question": cross_question},
        **update
    }


def needs_input(state: AgentState, node, message, **update):
    """The node can't go on with the current parameters; the user's next message changes them. Stays in the plan."""
    return {
        "last_result" : {"node": node, "status": "needs_input", "message": message},
        **clear_own_question(state, node),
        **update
    }


def failed(node, error):
    """A problem running the node again won't fix."""
    return {"last_result": {"node": node, "status": "error", "error": error, "recoverable": False}}


def earlier_history(state: AgentState):
    """History without the latest user message, which the nodes pass as the query."""
    return list(state.get("history") or [])[:-1]

In [ ]:
# fetch_trip node
from datetime import timedelta

from fetch_trips.fetch_trips import fetch_trip_details, fetch_trips_with_expiry, parse_utc

MAX_TRIPS_TO_PICK = 10          # more trips than this: ask the user to narrow them down first
MAX_PICK_WINDOW_DAYS = 2


def trip_label(trip):
    return f"{trip['driverName']} · {trip['assetId']} · {trip['startTimeUTC']}"


def select_trip(state: AgentState, trip_id, driver_id, trips):
    """Loads the full details of the chosen trip for the nodes after this one."""
    details = fetch_trip_details(state["fleet_id"], state["client_id"], driver_id, trip_id)
    return done(state, "fetch_trip", message=f"Selected trip {trip_id}.", data=details,
                trip_id=trip_id, selected_trip=details, trips=trips)


def narrow_down(state: AgentState, trips, driver_id, asset_id, has_window):
    """Too many trips to pick one: asks for a driver / asset and / or a shorter time range, None if both are set."""
    date_time = state.get("date_time")
    asks, purposes, options = [], [], {}

    if not (driver_id or asset_id):
        asks.append("pick a driver or an asset")
        purposes.append("driver_name or asset_name")
        # trips only have the asset ID, the user picks by name
        asset_names = {a["assetId"]: a["assetName"] for a in get_asset(fleet_id=state["fleet_id"], client_id=state["client_id"])}
        options = {
            "drivers" : sorted({t["driverName"] for t in trips if t["driverName"] not in (None, "—")}),
            "assets" : sorted({asset_names.get(t["assetId"]) or t["assetId"] for t in trips if t["assetId"] != "—"})
        }

    window_too_long = not has_window or (
        parse_utc(date_time.endTimeUTC) - parse_utc(date_time.startTimeUTC) > timedelta(days=MAX_PICK_WINDOW_DAYS))
    if window_too_long:
        asks.append(f"limit the time range to {MAX_PICK_WINDOW_DAYS} days")
        purposes.append("date_time")

    if not asks:
        return None
    question = f"I found {len(trips)} trips, too many to pick from. Please {' or '.join(asks)}."
    return ask(state, "fetch_trip", question, options=options, purpose=", ".join(purposes), trips=trips)


@graph_node("fetch_trip")
def fetch_trip_node(state: AgentState):
    """
    Fetches the trips matching the filters. When a later node needs one trip (DVR, QnA, previous
    requests), makes sure exactly one is selected: picks it if only one matches, asks the user to
    pick one if 2-10 match, and to narrow the filters down first if more than 10 match.
    """
    node = "fetch_trip"
    plan = list(state.get("plan") or [])
    needs_one_trip = any(n in TRIP_NODES for n in plan[1:])
    driver, asset, date_time = state.get("driver"), state.get("asset"), state.get("date_time")
    driver_id = driver.driver_id if driver else None
    asset_id = asset.asset_id if asset else None
    has_window = bool(date_time and date_time.startTimeUTC and date_time.endTimeUTC)
    limit_to_latest = state.get("limit_to_latest")
    trip_id = state.get("trip_id")

    # the user picked one of the trips this node offered (extract normally copies it to trip_id too)
    question, answer = state.get("cross_question"), own_answer(state, node)
    if answer and question.get("purpose") == "trip_id":
        picked = next((t for t in state.get("trips") or [] if answer in (t["tripId"], trip_label(t))), None)
        if picked:
            trip_id = picked["tripId"]

    # the user gave a trip ID, or picked one of the trips offered earlier
    if trip_id:
        known = next((t for t in state.get("trips") or [] if t["tripId"] == trip_id), None)
        trip_driver = known["driverId"] if known else driver_id
        if trip_driver:
            return select_trip(state, trip_id, trip_driver, state.get("trips") or [])
        if not (has_window or limit_to_latest):
            drivers = [d["driverName"] for d in get_drivers(fleet_id=state["fleet_id"], client_id=state["client_id"]) if d["driverName"]]
            return ask(state, node, f"Which driver drove trip {trip_id}?", options=drivers, purpose="driver_name")
        # otherwise look for it in the trips matching the filters

    if not (has_window or limit_to_latest):
        return needs_input(state, node, "Which trips should I look at? Give a date or date range, or ask for the latest trips.")

    event_type = state.get("event_type")
    try:
        trips = fetch_trips_with_expiry(
            state["fleet_id"], state["client_id"],
            start_time = date_time.startTimeUTC if has_window else None,
            end_time = date_time.endTimeUTC if has_window else None,
            driver_ids = [driver_id] if driver_id else None,
            asset_ids = [asset_id] if asset_id else None,
            events = {event_type: None} if event_type else None,
            limit_to_latest = None if has_window else limit_to_latest
        )
    except ValueError as e:
        # a missing or future date range, the user has to change it
        return needs_input(state, node, str(e))

    if trip_id:
        trip = next((t for t in trips if t["tripId"] == trip_id), None)
        if trip:
            return select_trip(state, trip_id, trip["driverId"], trips)
        return needs_input(state, node, f"I couldn't find trip {trip_id} for the selected filters. "
                                        "Which driver or date is it from?", trips=trips)

    if not trips:
        return needs_input(state, node, "No trips found for the selected filters. Try a different date range, driver or asset.", trips=[])

    # only listing trips
    if not needs_one_trip:
        return done(state, node, message=f"Found {len(trips)} trips.", data=trips, trips=trips)

    if len(trips) == 1:
        return select_trip(state, trips[0]["tripId"], trips[0]["driverId"], trips)

    if len(trips) > MAX_TRIPS_TO_PICK:
        narrow = narrow_down(state, trips, driver_id, asset_id, has_window)
        if narrow:
            return narrow

    options = [{"tripId": t["tripId"], "label": trip_label(t)} for t in trips]
    return ask(state, node, f"I found {len(trips)} trips. Which one do you mean?",
               options=options, purpose="trip_id", trips=trips)

In [ ]:
# time_slot node
from time_slot.time_slot_agent import run_time_slot_agent


@graph_node("time_slot")
def time_slot_node(state: AgentState):
    """Finds the moment of the selected trip the footage should cover, asking the user when it's unclear."""
    node = "time_slot"
    trip = state.get("selected_trip")
    if not trip:
        return failed(node, "No trip is selected to find the time slot in.")

    query = state["query"]
    question = state.get("cross_question")
    if own_answer(state, node) and question.get("user_feedback") is not None:
        query = f"{query}\nThe user picked: {own_answer(state, node)}"

    result = run_time_slot_agent(query, trip, history=earlier_history(state))

    if result["status"] == "success":
        update = done(state, node, message=result["thinking"], data=result["time_slot"], time_slot=result["time_slot"])
        update["last_result"]["thinking"] = result["thinking"]
        return update

    update = ask(state, node, result["message"] or result["thinking"], options=result["options"],
                 purpose="the moment of the trip the footage should cover")
    update["last_result"]["thinking"] = result["thinking"]
    return update

In [ ]:
# trip_qna node
from question_trips.QnA import answer_trip_query
from time_slot.time_slot_util import simplify_response


@graph_node("trip_qna")
def trip_qna_node(state: AgentState):
    """Answers the user's question about the selected trip."""
    node = "trip_qna"
    trip = state.get("selected_trip")
    if not trip:
        return failed(node, "No trip is selected to answer the question about.")

    result = answer_trip_query(state["query"], simplify_response(trip), history=earlier_history(state))
    return done(state, node, message=result["answer"], data={"category": result["category"]})

In [ ]:
# prev_upload_requests node
from prev_upload_requests.get_prev_requests import get_trip_upload_requests


@graph_node("prev_upload_requests")
def prev_upload_requests_node(state: AgentState):
    """Lists the DVR clip / timelapse requests already made for the selected trip."""
    node = "prev_upload_requests"
    trip = state.get("selected_trip")
    if not trip:
        return failed(node, "No trip is selected to list the upload requests of.")

    upload_requests, message = get_trip_upload_requests(trip)
    return done(state, node, message=message, data=upload_requests or [])

In [ ]:
# dvr node
from upload_request.create_request import REQUEST_TYPES, create_dvr_request

CLIP_TYPES = {"dvr_clip": "clip", "timelapse": "timelapse"}     # extract / time slot names -> create_request names


def iso_utc(time_utc):
    """'2026-06-05 16:20:00' (time slot) -> '2026-06-05T16:20:00Z' (create_request)."""
    time_utc = time_utc.replace(" ", "T")
    return time_utc if time_utc.endswith("Z") else time_utc + "Z"


@graph_node("dvr")
def dvr_node(state: AgentState):
    """
    Shows the DVR clip / timelapse params for the user to confirm, then places the request.
    The user can change the video format / resolution instead of confirming, it asks again with them.
    """
    node = "dvr"
    trip, time_slot = state.get("selected_trip"), state.get("time_slot")
    if not trip or not time_slot:
        return failed(node, "A DVR request needs a selected trip and its time slot.")

    params = {
        "tripId" : trip["tripId"],
        "driverId" : trip["driverId"],
        "assetId" : trip.get("asset", {}).get("assetId"),
        "type" : CLIP_TYPES[state.get("clip_type") or "dvr_clip"],
        "clipStart" : iso_utc(time_slot["startTimeUTC"]),
        "clipEnd" : iso_utc(time_slot["endTimeUTC"])
    }
    if state.get("video_format"):
        params["videoFormat"] = state["video_format"]
    if state.get("video_resolution"):
        params["videoResolution"] = state["video_resolution"]

    result = create_dvr_request(state["fleet_id"], state["client_id"], params, user_response=own_answer(state, node))

    if result["status"] == "submitted":
        request_type = REQUEST_TYPES[result["params"]["type"]]
        return done(state, node, message=f"{request_type} request placed. Request ID: {result['uploadRequestId']}",
                    data=result, dvr_params=result["params"])

    if result["status"] == "invalid":
        # the clip times come from the time slot, so it runs again with the user's correction
        plan = list(state.get("plan") or [])
        return needs_input(state, node, result["message"], plan=["time_slot", *plan], dvr_params=result["params"])

    return ask(state, node, result["message"], options=["confirm"],
               purpose="confirm the DVR request params", dvr_params=result["params"])

In [ ]:
Can you uh, test the execution of extract agent node? Uh, by using the fleet ID as lmfleet003 client ID as uh, lmdemotsp and uh, for multiple type of queries where first when there is a cross question second when there is need to be a cross question third for a simple query where uh, parameters need to be extracted, make real requests
